# 🎯 Answer Accuracy Evaluation — Showcase Notebook

This notebook is a **portfolio-grade demonstration** of GenAI answer-accuracy evaluation. It is designed to show not only that a model can be scored, but that the evaluation is thoughtfully framed, reproducible, interpretable, and extensible.

The notebook answers six questions:

1. **What model are we testing?** Target model and provider are confirmed before the run.
2. **What datasets are we testing on?** Dataset preset, selected datasets, splits, cache paths, and sample sizes are shown explicitly.
3. **What kind of tasks are these?** Different datasets have different structures: multiple-choice, open-domain QA, context-grounded QA, multi-hop QA, truthfulness-style generation.
4. **What metrics are used?** Exact match, contains match, token F1, TF-IDF similarity, and composite score are explained with limitations.
5. **Where did the model struggle?** Diagnostics flag weak, borderline, and metric-disagreement cases with reasons.
6. **What can be reported?** An in-notebook HTML executive report summarizes target model, datasets, pass rate, flagged cases, and artifacts.

Heavy implementation lives in `src/genai_capability_bench`; the notebook is the guided evaluation and communication layer.


---

## 🧭 1. Evaluation Framing

**Capability under test:** closed-book answer accuracy.

Answer accuracy tests whether a model can produce an expected factual answer. It is one of the most basic LLM capabilities, but it has important design choices:

| Design Question | Why It Matters |
|---|---|
| Closed-book or context-grounded? | Closed-book tests model knowledge; context-grounded tests reading and grounding. |
| Short answer or multiple choice? | Short answer needs reference/semantic matching; multiple choice should use exact option accuracy. |
| Public benchmark or custom golden set? | Public datasets support comparability; custom sets support business relevance. |
| Rule-based metric or judge model? | Rule-based metrics are reproducible; judge models help with ambiguity but add cost and variability. |

This notebook starts with transparent rule-based metrics and supports optional judge review only for flagged cases.


---

## 🗃️ 2. Benchmark Dataset Strategy

This notebook treats dataset selection as a benchmark-design decision, not a file-loading detail. Each preset maps to a distinct evaluation purpose:

| Preset | Datasets | Primary Question |
|---|---|---|
| `local_smoke` | `answer_accuracy_sample` | Does the workflow run correctly with a small local golden set? |
| `broad_knowledge` | `mmlu` | How does the model perform across broad academic and professional subjects? |
| `open_domain_qa` | `triviaqa`, `natural_questions` | Can the model answer open-domain factual questions with acceptable reference alignment? |
| `science_exam` | `arc` | Can the model handle science exam-style factual reasoning? |
| `context_grounded` | `squad` | Can the model answer from supplied context? This overlaps with future RAG evaluation. |
| `multi_hop` | `hotpotqa` | Can the model answer questions that often require connecting multiple facts? This overlaps with reasoning/RAG. |
| `custom_golden_set` | `custom` | How does the model perform on organization-specific questions that matter in practice? |

The public datasets are normalized into a shared `EvalTask` schema, but their native structures differ. That matters because scoring a multiple-choice exam, a short open-domain answer, and a context-grounded answer should not be interpreted the same way.

Operational guidance:

- Start with `local_smoke` before using public datasets or paid model calls.
- For public datasets, use `SAMPLE_SIZE_PER_DATASET=10` to validate the full path and `25-100` for exploratory comparisons.
- `SAMPLE_SIZE_PER_DATASET=None` means full selected split. This is blocked by default for public datasets because it can trigger tens of thousands of model calls.
- `MAX_TASKS_PER_RUN` is a final safety cap. Raise it only for a planned large benchmark with budget/runtime approval.
- Keep `DATASET_KEYS` explicit when you need full control; presets are just convenient benchmark portfolios.


---

## 📏 3. Metrics and Interpretation

The current scoring stack is intentionally transparent:

- **Exact Match**: normalized prediction exactly equals a reference answer.
- **Contains Match**: reference answer appears inside the model answer.
- **Token F1**: token overlap between prediction and reference.
- **TF-IDF Similarity**: lightweight local similarity signal for wording variation.
- **Composite Score**: maximum of exact match, contains match, and a weighted token/TF-IDF blend.

What is **not** currently used in this notebook:

- BLEU: useful for translation-like n-gram overlap, but often poor for factual QA.
- ROUGE: useful for summarization overlap, but not ideal for short factual answers.
- Embedding similarity: planned as a stronger semantic metric.
- LLM-as-judge: optional for flagged cases, not the default primary scorer.

Interpretation rule of thumb:

| Score | Interpretation |
|---|---|
| 0.90 - 1.00 | Strong match for short factual QA |
| 0.70 - 0.89 | Likely acceptable; review in high-stakes settings |
| 0.40 - 0.69 | Partial/uncertain; review recommended |
| 0.00 - 0.39 | Likely incorrect or unsupported |


In [ ]:
# =============================================================================
# CELL 1: IMPORTS & SETUP
# =============================================================================
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings('ignore')

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML

try:
    import ipywidgets as widgets
    WIDGETS_IMPORTABLE = True
except Exception:
    WIDGETS_IMPORTABLE = False

from genai_capability_bench.core.runner import config_float, load_config, parse_models
from genai_capability_bench.reporting.diagnostics import metric_guide_table, score_interpretation_table
from genai_capability_bench.reporting.notebook_views import (
    dataset_catalog_table,
    dataset_preset_table,
    html_summary_report,
    judge_config_table,
    model_config_table,
    provider_environment_table,
    sample_size_guidance_table,
    selected_dataset_plan_table,
)
from genai_capability_bench.reporting.plots import plot_capability_scores
from genai_capability_bench.workflows.answer_accuracy import (
    AnswerAccuracyRunConfig,
    load_answer_accuracy_tasks,
    run_answer_accuracy_workflow,
)

OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'runs'

print('✅ Notebook environment ready')
print(f'📁 Repo root: {REPO_ROOT}')


In [ ]:
# =============================================================================
# CELL 2: CATALOGS — DATASETS, PRESETS, SAMPLE SIZE, AND METRICS
# =============================================================================
print('🧭 Dataset presets — benchmark purpose and recommended starting scale')
display(dataset_preset_table())

print('📐 Sample-size guidance — confidence, cost, and runtime tradeoff')
display(sample_size_guidance_table())

print('📚 Registered dataset catalog — normalized into the shared EvalTask schema')
display(dataset_catalog_table())

print('📏 Metric methodology')
display(metric_guide_table())

display(score_interpretation_table())


---

## ⚙️ 4. Configuration — Benchmark Controls

This is the main control surface for the run. A non-technical reviewer should be able to read this section and understand **what model**, **what datasets**, **how many samples**, and **what review policy** are being used.

### Model Config

- `eval_openai_compatible_template.yaml` reads provider settings from `.env` and is intended for Azure OpenAI, OpenAI, Ollama, Groq, Together AI, LM Studio, and other OpenAI-compatible endpoints.
- `eval_core_demo.yaml` runs the local mock model and is useful for validating the workflow without API cost.
- Some GPT-5/reasoning deployments reject `temperature`, `max_tokens`, or similar generation controls. Set `OPENAI_TEMPERATURE=omit`, `OPENAI_MAX_TOKENS=omit`, and `OPENAI_TOKEN_PARAMETER=` in `.env`, or leave those variables undefined when using this template.

### Dataset and Run Size Controls

Most users only need to choose two things:

- `DATASET_PRESET`: the benchmark portfolio, such as `local_smoke`, `broad_knowledge`, or `open_domain_qa`.
- `RUN_MODE`: the run size and safety posture. Use `quick_check`, `exploratory`, `report_sample`, or `full_public`.

How to interpret the derived settings:

- **Sample per dataset** means the notebook samples up to that many rows from each selected dataset. For `open_domain_qa`, `100` means up to 100 TriviaQA rows plus up to 100 Natural Questions rows.
- **Estimated model calls** means sampled tasks × number of target models. This is the practical cost/runtime driver.
- **Safety cap** stops the run if the materialized task count is unexpectedly larger than planned.
- **Full public split** means the entire selected public benchmark split. It only happens in `RUN_MODE = 'full_public'`; otherwise full-split loading is blocked.
- **Checkpoint every** controls how often resumable progress is written to disk.

Advanced users can still override splits, categories, checkpoint paths, and custom datasets below.

### Judge Review

Judge review is disabled by default. When enabled, it is used for flagged/ambiguous cases, not as the primary score. This keeps the benchmark reproducible while still supporting expert-style review where rule-based metrics are weak.


In [ ]:
# =============================================================================
# CELL 3: CONFIGURATION BLOCK — edit this cell
# =============================================================================
USE_INTERACTIVE_WIDGETS = False  # Set True only if ipywidgets renders correctly in your notebook frontend.

# Target model config:
MODEL_CONFIG_PATH = REPO_ROOT / 'configs' / 'eval_openai_compatible_template.yaml'
# MODEL_CONFIG_PATH = REPO_ROOT / 'configs' / 'eval_core_demo.yaml'

DATASET_PRESETS = {
    'local_smoke': ['answer_accuracy_sample'],
    'broad_knowledge': ['mmlu'],
    'open_domain_qa': ['triviaqa', 'natural_questions'],
    'science_exam': ['arc'],
    'context_grounded': ['squad'],
    'multi_hop': ['hotpotqa'],
    'custom_golden_set': ['custom'],
}

# Choose the benchmark portfolio.
DATASET_PRESET = 'open_domain_qa'
DATASET_KEYS = DATASET_PRESETS[DATASET_PRESET]

# Choose the run size. This is the main control most users should change.
# - quick_check: tiny run to verify the pipeline and credentials.
# - exploratory: useful first benchmark signal without a long/costly run.
# - report_sample: larger sampled run for a more credible notebook/report artifact.
# - full_public: full selected public split; use only when you intentionally want the entire benchmark.
RUN_MODE = 'exploratory'

RUN_PROFILES = {
    'quick_check': {
        'sample_size_per_dataset': 10,
        'max_tasks_per_run': 50,
        'allow_full_public_dataset': False,
        'guidance': 'Fast path check. Good before any expensive run.',
    },
    'exploratory': {
        'sample_size_per_dataset': 100,
        'max_tasks_per_run': 500,
        'allow_full_public_dataset': False,
        'guidance': 'Recommended first real public benchmark run.',
    },
    'report_sample': {
        'sample_size_per_dataset': 500,
        'max_tasks_per_run': 2500,
        'allow_full_public_dataset': False,
        'guidance': 'Larger sampled run for stronger analysis; watch cost/runtime.',
    },
    'full_public': {
        'sample_size_per_dataset': None,
        'max_tasks_per_run': None,
        'allow_full_public_dataset': True,
        'guidance': 'Runs the full selected public split. Use only with budget/runtime approval.',
    },
}

profile = RUN_PROFILES[RUN_MODE]
SAMPLE_SIZE_PER_DATASET = profile['sample_size_per_dataset']
MAX_TASKS_PER_RUN = profile['max_tasks_per_run']
ALLOW_FULL_PUBLIC_DATASET = profile['allow_full_public_dataset']

# Advanced overrides — usually leave these unchanged.
DATASET_SPLITS = {}      # e.g., {'mmlu': 'test', 'squad': 'validation'}
CUSTOM_DATASET_PATH = None
SELECTED_CATEGORIES = 'ALL'
RANDOM_SEED = 42
CHECKPOINT_EVERY = 50
CHECKPOINT_DIR = None  # Defaults to outputs/runs/_checkpoints/<run_id>/
RESUME_FROM_CHECKPOINT = None  # Paste a prior checkpoint folder/file here to continue an interrupted run.

DOWNLOAD_IF_MISSING = True
CACHE_LOCAL_COPY = True
REFRESH_DATASET_CACHE = False

PASS_THRESHOLD = None
RUN_ID_PREFIX = 'answer_accuracy_demo'

# Optional judge review for flagged cases only.
ENABLE_JUDGE_REVIEW = False
JUDGE_MODEL_NAME = os.environ.get('OPENAI_JUDGE_MODEL')
JUDGE_MAX_CASES = 10

if USE_INTERACTIVE_WIDGETS and WIDGETS_IMPORTABLE:
    preset_widget = widgets.Dropdown(options=list(DATASET_PRESETS.keys()), value=DATASET_PRESET, description='Preset')
    mode_widget = widgets.Dropdown(options=list(RUN_PROFILES.keys()), value=RUN_MODE, description='Run mode')
    judge_widget = widgets.Checkbox(value=ENABLE_JUDGE_REVIEW, description='Enable judge review')
    display(widgets.VBox([preset_widget, mode_widget, judge_widget]))
    DATASET_PRESET = preset_widget.value
    DATASET_KEYS = DATASET_PRESETS[DATASET_PRESET]
    RUN_MODE = mode_widget.value
    profile = RUN_PROFILES[RUN_MODE]
    SAMPLE_SIZE_PER_DATASET = profile['sample_size_per_dataset']
    MAX_TASKS_PER_RUN = profile['max_tasks_per_run']
    ALLOW_FULL_PUBLIC_DATASET = profile['allow_full_public_dataset']
    ENABLE_JUDGE_REVIEW = bool(judge_widget.value)
elif USE_INTERACTIVE_WIDGETS and not WIDGETS_IMPORTABLE:
    print('ipywidgets is not installed. Falling back to manual variables above.')

model_config = load_config(MODEL_CONFIG_PATH)
models = parse_models(model_config)

planned_calls = (
    'FULL SELECTED SPLIT'
    if SAMPLE_SIZE_PER_DATASET is None
    else SAMPLE_SIZE_PER_DATASET * len(DATASET_KEYS) * len(models)
)
sample_label = 'Full selected split' if SAMPLE_SIZE_PER_DATASET is None else f'{SAMPLE_SIZE_PER_DATASET} per dataset'
cap_label = 'No cap' if MAX_TASKS_PER_RUN is None else f'{MAX_TASKS_PER_RUN} model calls'

workflow_config = AnswerAccuracyRunConfig(
    repo_root=REPO_ROOT,
    model_config_path=MODEL_CONFIG_PATH,
    dataset_keys=DATASET_KEYS,
    dataset_splits=DATASET_SPLITS,
    sample_size_per_dataset=SAMPLE_SIZE_PER_DATASET,
    selected_categories=SELECTED_CATEGORIES,
    custom_dataset_path=CUSTOM_DATASET_PATH,
    random_seed=RANDOM_SEED,
    pass_threshold=PASS_THRESHOLD,
    run_id_prefix=RUN_ID_PREFIX,
    output_root=OUTPUT_ROOT,
    download_if_missing=DOWNLOAD_IF_MISSING,
    cache_local_copy=CACHE_LOCAL_COPY,
    refresh_dataset_cache=REFRESH_DATASET_CACHE,
    enable_judge_review=ENABLE_JUDGE_REVIEW,
    judge_model_name=JUDGE_MODEL_NAME,
    judge_max_cases=JUDGE_MAX_CASES,
    max_tasks_per_run=MAX_TASKS_PER_RUN,
    allow_full_public_dataset=ALLOW_FULL_PUBLIC_DATASET,
    checkpoint_every=CHECKPOINT_EVERY,
    checkpoint_dir=CHECKPOINT_DIR,
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
)

print('✅ Configuration loaded')
print(f'📌 Dataset preset: {DATASET_PRESET}')
print(f'📚 Dataset keys: {DATASET_KEYS}')
print(f'🚦 Run mode: {RUN_MODE} — {profile["guidance"]}')
print(f'🧪 Sample plan: {sample_label}')
print(f'📞 Planned model calls before filtering: {planned_calls}')
print(f'🛑 Safety cap: {cap_label}')
print(f'🌐 Full public split allowed: {ALLOW_FULL_PUBLIC_DATASET}')
print(f'💾 Checkpoint every: {CHECKPOINT_EVERY} completed model calls')
print(f'🔁 Resume from checkpoint: {RESUME_FROM_CHECKPOINT or "None"}')
print(f'⚖️ Judge review enabled: {ENABLE_JUDGE_REVIEW}')

display(pd.DataFrame([{
    'Run Mode': RUN_MODE,
    'Datasets Selected': len(DATASET_KEYS),
    'Sample Per Dataset': sample_label,
    'Estimated Model Calls': planned_calls,
    'Safety Cap': cap_label,
    'Full Public Split': ALLOW_FULL_PUBLIC_DATASET,
    'Checkpoint Every': CHECKPOINT_EVERY,
}]))

print('🧾 Active dataset plan')
display(selected_dataset_plan_table(
    DATASET_KEYS,
    preset_name=DATASET_PRESET,
    sample_size_per_dataset=SAMPLE_SIZE_PER_DATASET,
    dataset_splits=DATASET_SPLITS,
    selected_categories=SELECTED_CATEGORIES,
))


In [ ]:
# =============================================================================
# CELL 4: TARGET MODEL AND JUDGE CONFIRMATION
# =============================================================================
print('🎯 Target model configuration')
display(model_config_table(models))

print('🔌 Provider environment — secrets redacted')
display(provider_environment_table())

print('🧑‍⚖️ Optional judge model configuration')
display(judge_config_table())

if any(m.provider == 'mock' for m in models):
    display(Markdown('> **Mode:** This run is using `DemoMock`. It validates the workflow but does not benchmark a real LLM.'))
else:
    model_names = ', '.join(m.model for m in models)
    display(Markdown(f'> **Target LLM confirmed:** `{model_names}` via `{models[0].provider}`.'))

if ENABLE_JUDGE_REVIEW:
    display(Markdown(f'> **Judge review enabled:** flagged cases will be reviewed by `{JUDGE_MODEL_NAME}`.'))
else:
    display(Markdown('> **Judge review disabled:** the run will use deterministic metrics and flag cases for optional review.'))


---

## ✅ 5. Dataset Materialization and Pre-Flight Check

This section confirms the benchmark before any model calls are made. It is intentionally explicit because dataset choice is part of the evaluation evidence.

The **Dataset Manifest** answers four practical questions:

- Which dataset key and split were selected?
- Was the source local, cached, custom, or downloaded from the public dataset source?
- How many raw tasks were loaded and how many answer-accuracy tasks were retained?
- Which normalized categories are represented in the selected sample?

If the manifest does not match your intent, stop and adjust `DATASET_PRESET`, `DATASET_KEYS`, `DATASET_SPLITS`, `SELECTED_CATEGORIES`, `SAMPLE_SIZE_PER_DATASET`, or `MAX_TASKS_PER_RUN` before running the evaluation. A full public split can be very large and is blocked unless `ALLOW_FULL_PUBLIC_DATASET=True`.


In [ ]:
# =============================================================================
# CELL 5: PRE-FLIGHT CHECK
# =============================================================================
tasks, dataset_manifest_df = load_answer_accuracy_tasks(workflow_config)

preflight_df = pd.DataFrame([
    {'Check': 'Model config exists', 'Status': MODEL_CONFIG_PATH.exists(), 'Detail': str(MODEL_CONFIG_PATH)},
    {'Check': 'Target model configured', 'Status': len(models) > 0, 'Detail': ', '.join(m.model for m in models)},
    {'Check': 'Dataset preset selected', 'Status': bool(DATASET_PRESET), 'Detail': DATASET_PRESET},
    {'Check': 'Dataset keys selected', 'Status': len(DATASET_KEYS) > 0, 'Detail': ', '.join(DATASET_KEYS)},
    {'Check': 'Answer-accuracy tasks available', 'Status': len(tasks) > 0, 'Detail': f'{len(tasks)} tasks'},
    {'Check': 'Output directory available', 'Status': OUTPUT_ROOT.exists() or OUTPUT_ROOT.parent.exists(), 'Detail': str(OUTPUT_ROOT)},
])

display(preflight_df)

print('📦 Dataset Manifest — what will be evaluated')
display(dataset_manifest_df)

category_df = pd.DataFrame({'category': [t.category for t in tasks]}).value_counts().reset_index(name='n')
print('🏷️ Selected category distribution')
display(category_df)

if not all(preflight_df['Status']):
    raise RuntimeError('❌ Pre-flight failed. Fix failed checks before continuing.')

print('✅ Pre-flight passed. The next cell will run model evaluation.')


---

## 🚀 6. Run Evaluation

This cell runs the answer-accuracy workflow. For real LLMs, this is where API calls begin. The progress bar shows completed model calls and percentage progress.

What gets saved:

- raw normalized results
- category summary
- dataset/category summary
- leaderboard
- diagnostics with flag reasons
- markdown report
- dataset manifest
- resumable checkpoint snapshots under `outputs/runs/_checkpoints/<run_id>/`


In [ ]:
# =============================================================================
# CELL 6: RUN EVALUATION
# =============================================================================
run = run_answer_accuracy_workflow(workflow_config, show_progress=True)

print('✅ Evaluation complete')
print(f'📁 Run directory: {run.run_dir}')
print(f'📄 Markdown report: {run.report_path}')
print(f'💾 Checkpoint path: {run.checkpoint_path}')

display(run.results_df[['model_name', 'dataset_key', 'task_id', 'category', 'actual_output', 'expected_output', 'score', 'passed']])


---

## 🔎 7. Result Analysis and Diagnostics

A score alone is not enough. The diagnostic table explains **what was flagged and why**.

Flag categories:

| Flag | Meaning |
|---|---|
| High review priority | Score is very low; likely incorrect or missing answer. |
| Near threshold | Score is below pass threshold but not catastrophic. |
| Metric disagreement | Metrics disagree strongly; may need semantic or judge review. |
| Looks good | No immediate issue triggered. |

If judge review is enabled, flagged cases may also include `judge_score` and `judge_reason`.


In [ ]:
# =============================================================================
# CELL 7: RESULT ANALYSIS AND DIAGNOSTICS
# =============================================================================
print('📋 Category summary')
display(run.summary_df)

print('🗃️ Dataset/category summary')
display(run.dataset_summary_df)

print('🏁 Leaderboard')
display(run.leaderboard_df)

print('🚩 Flagged cases and reasons')
review_cols = [
    'flagged', 'review_priority', 'flag_reason', 'recommended_action',
    'model_name', 'dataset_key', 'task_id', 'category', 'actual_output',
    'expected_output', 'score', 'strict_score', 'contains_only_credit',
    'exact_match', 'contains_match', 'token_f1', 'tfidf_similarity',
    'metric_disagreement'
]
optional_cols = [c for c in ['judge_score', 'judge_reason'] if c in run.diagnostics_df.columns]
review_cols = review_cols + optional_cols
flagged_df = run.diagnostics_df[run.diagnostics_df['flagged']].copy()

if flagged_df.empty:
    display(Markdown('✅ No cases were flagged by the current diagnostic rules.'))
else:
    display(flagged_df.sort_values(['score', 'task_id'])[review_cols])

print('Full diagnostics table')
display(run.diagnostics_df.sort_values(['score', 'task_id'])[review_cols])


---

## 📊 8. Visualization

These charts are designed to make the evaluation story visible quickly:

- category score profile
- score distribution
- review-priority mix
- dataset/category performance


In [ ]:
# =============================================================================
# CELL 8: VISUALIZATION
# =============================================================================
plot_capability_scores(run.summary_df, title='Answer Accuracy by Category');

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(run.results_df['score'], bins=10, color='#2563eb', alpha=0.85, edgecolor='white')
threshold = workflow_config.pass_threshold or config_float(model_config.get('default_pass_threshold'), 0.7)
axes[0].axvline(threshold, color='#dc2626', linestyle='--', label=f'Pass threshold ({threshold:.2f})')
axes[0].set_title('Score Distribution')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Responses')
axes[0].set_xlim(0, 1)
axes[0].grid(axis='y', alpha=0.25)
axes[0].legend()

priority_counts = run.diagnostics_df['review_priority'].value_counts()
axes[1].bar(priority_counts.index, priority_counts.values, color='#0f766e')
axes[1].set_title('Review Priority Mix')
axes[1].set_ylabel('Cases')
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(axis='y', alpha=0.25)

fig.tight_layout()
plt.show()


---

## 📝 9. Executive Report

The report below is rendered directly in the notebook for review. It is also saved as Markdown in the run folder.

For a more advanced future version, the deterministic report can be supplemented with optional LLM-generated narrative, clearly labeled as AI-generated analysis.


In [ ]:
# =============================================================================
# CELL 9: HTML EXECUTIVE REPORT
# =============================================================================
target_label = ', '.join(f'{m.name} / {m.model}' for m in models)
display(HTML(html_summary_report(run, target_label=target_label, judge_enabled=ENABLE_JUDGE_REVIEW)))

print('📦 Saved artifacts')
for path in sorted(run.run_dir.iterdir()):
    print(f'- {path.name}')


---

## 🧠 10. Interpretation and Next Steps

This notebook demonstrates the answer-accuracy slice of a broader capability evaluation suite.

Recommended next improvements:

1. Add exact multiple-choice grading for MMLU and ARC.
2. Add embedding similarity for open-ended factual QA.
3. Add optional judge narratives for flagged cases and executive reporting.
4. Add custom enterprise/domain golden sets.
5. Compare multiple target models under identical dataset selections.
6. Add regression testing against previous runs.

The mindset: evaluation is not just scoring. It is dataset design, metric design, diagnostic design, model-risk evidence, and communication.
